# Few-Shot Classification — UVA Kimi K2.5

Companion to `zeroshot_kimik2_uva_v2.ipynb`. Same UVA Kimi K2.5 endpoint and prompt base, but now few-shot:

- **Demo examples** are a stratified sample (4 per class, 8 total by default) drawn from `svm_train_80.xlsx` — the 80% training pool. This never overlaps with the eval set, so nothing the model is scored on has appeared as a demo.
- **Test set** is `eval_holdout_20_unlabeled.csv` (82 posts, no ground-truth labels) — the same file used for the zero-shot and SVM runs, so all three approaches are scored on identical data.

Runs with a small thread pool (`CONCURRENCY=4`) instead of one call at a time — 82 posts don't warrant a Slurm array job, just modest parallelism within a single job.

Same API key setup as before — `.env` file (or shell env var) with `UVARC_GenAI_API=sk-...` next to this notebook.

Make sure `eval_holdout_20_unlabeled.csv` and `svm_train_80.xlsx` are in the same folder as this notebook (or update the paths).

In [1]:
!pip3 install httpx pandas openpyxl --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
"""
Few-Shot Kimi K2.5 classification (UVA RC OpenWebUI).
Demo examples: stratified sample pulled from svm_train_80.xlsx (the 80%
training pool — never overlaps with the eval set).
Test set: eval_holdout_20_unlabeled.csv (82 posts, no ground-truth labels —
same file the zero-shot and SVM runs use, for an apples-to-apples comparison).
Runs with a small thread pool (CONCURRENCY workers) instead of one call at
a time — 82 posts don't need a Slurm array job, just modest parallelism.
"""

import os
import re
import sys
import json
import time
import httpx
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── CONFIG ────────────────────────────────────────────────────────────────
TEST_INPUT_PATH     = "eval_holdout_20_unlabeled.csv"   # id, text only — no Label column
FEWSHOT_SOURCE_PATH = "svm_train_80.csv"                 # has Label + label_reason
FEWSHOT_PER_CLASS    = 4                                 # demo examples per class (4+4 = 8 total)
FEWSHOT_RANDOM_STATE = 42

OUTPUT_PATH  = "preds_fewshot.csv"

TEXT_COL  = "text"
LABEL_COL = "Label"
ID_COL    = "id"

UVARC_BASE_URL = "https://open-webui.rc.virginia.edu/api"
UVARC_CHAT_ENDPOINT = f"{UVARC_BASE_URL}/chat/completions"
MODEL = "Kimi K2.5"

TEMPERATURE = 0.0
MAX_TOKENS  = 2000        # reasoning model burns tokens on chain-of-thought before output
MAX_RETRIES = 5
DEBUG_FIRST_N = 2         # print raw response for the first N calls so you can sanity-check parsing

CONCURRENCY = 4           # matches the known-good concurrency setting against the UVA endpoint
                           # (higher risks the shared endpoint's 400 "overload" error)


# ── .env loader (same pattern as before) ─────────────────────────────────────
def _load_env_file():
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        script_dir = os.getcwd()

    for candidate in [os.path.join(script_dir, ".env"), os.path.join(os.getcwd(), ".env")]:
        if os.path.exists(candidate):
            print(f"  Loading .env from: {candidate}")
            with open(candidate) as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    line = re.sub(r"^export\s+", "", line)
                    if "=" in line:
                        k, _, v = line.partition("=")
                        k = k.strip()
                        v = v.strip().strip('"').strip("'")
                        if k and k not in os.environ:
                            os.environ[k] = v
            return candidate
    return None


_found = _load_env_file()
if not _found:
    print("  (No .env file found — falling back to shell environment)")

UVARC_API_KEY = os.environ.get("UVARC_GenAI_API")
if not UVARC_API_KEY:
    sys.exit(
        "\nERROR: UVARC_GenAI_API not found.\n"
        "  Create a .env file next to this notebook containing:\n"
        "    UVARC_GenAI_API=sk-your-key-here\n"
    )
print("API key loaded (starts with):", UVARC_API_KEY[:8] + "...")


In [ ]:
# ── Prompt templates (zero-shot header + few-shot wrapper, same as original) ─
BASE_HEADER = """You are an expert in labeling burnout-related Reddit posts from cybersecurity professionals.

Classify the post into ONE of the following categories:

0 (out-of-scope): Not about work stress/burnout at all — pure technical questions, news, product discussions, memes, or other unrelated content.

1-9 (in-scope): The post relates to work-related burnout or stress in one of these specific ways:
  1 = Work-related burnout / chronic stress / workload / work pressure (job demands, long hours, on-call, exhaustion from work, etc.)
  2 = Toughest situation — could be work related; a hint of possible stress, work-related frustration, mental exhaustion, or depression
  3 = Imposter syndrome in work life; not knowing something work-related; feeling like an outsider/pariah
  4 = PTSD, mental health disorder/illness, bipolar disorder, ADHD, schizophrenia
  5 = De-stress / stress relief / trying not to become overwhelmed
  6 = Work-life balance, work-related health, workload balance
  7 = Job search / interview / hard time getting a job / job change / job confusion / job tasks feel slow or not getting it / not feeling good enough about job / job market burnout / trust issues at work-ethical stress / toxic environment
  8 = Anxiety
  9 = Staying motivated, learning new things

IMPORTANT — final label rule: Categories 1 through 9 are ALL in-scope. Regardless of which specific subcategory (1-9) applies, the final "label" field must be 1. Only use "label": 0 for posts that are genuinely out-of-scope (category 0). The "subcategory" field is where you record which specific number (0-9) applies — this is for reasoning/audit purposes and is separate from "label".

Emphasis & Caution: Hypothetical or purely future/imaginary burnout scenarios where the poster is not describing their own past or present experience should lean toward out-of-scope (0) unless another in-scope category clearly applies (e.g. job search anxiety about a future interview is still in-scope under category 7/8).

Respond ONLY in this exact JSON format (no other text, no markdown, no code fences):
{
  "subcategory": 0-9,
  "label": 0 or 1,
  "label_reason": "1-2 sentences citing the specific language in the post that supports this category"
}"""


def build_few_shot_prompt(text, fewshot_examples_text):
    return f"""{BASE_HEADER}

Here are some labeled examples:

{fewshot_examples_text}

Now classify the following post:
\"\"\"
{text}
\"\"\"

JSON:"""


def create_fewshot_examples_text(fewshot_df):
    """
    Uses the existing label_reason text from labeled_dataset_new.xlsx as the
    reasoning field in each example. If a subcategory number appears in
    parentheses in label_reason (e.g. "Job search (7): ..."), it's extracted;
    otherwise subcategory falls back to the collapsed label (0 or 1).
    """
    examples = []
    for _, row in fewshot_df.iterrows():
        label = 1 if row[LABEL_COL] == 1 else 0
        reason = str(row.get("label_reason", "")).strip()

        subcat_match = re.search(r"\((\d)(?:/\d)?\)", reason)
        subcategory = int(subcat_match.group(1)) if subcat_match else label

        reason_escaped = reason.replace("\\", "\\\\").replace('"', '\\"')

        example_json = (
            f'{{"subcategory": {subcategory}, "label": {label}, '
            f'"label_reason": "{reason_escaped}"}}'
        )
        examples.append(
            f"Post:\n\"\"\"\n{row[TEXT_COL]}\n\"\"\"\nJSON:\n{example_json}"
        )
    return "\n\n".join(examples)

In [4]:
# ── UVA Kimi K2.5 call (raw httpx, defensive parsing — identical to zero-shot notebook) ─
_call_counter = {"n": 0}


def _extract_text_from_response(raw_text):
    raw_text = raw_text.strip()

    # 1) Plain JSON body, OpenAI-style: {"choices":[{"message":{"content":...}}]}
    try:
        data = json.loads(raw_text)
        choice = data["choices"][0]
        if "message" in choice and "content" in choice["message"]:
            return choice["message"]["content"]
        if "text" in choice:
            return choice["text"]
    except Exception:
        pass

    # 2) SSE stream: lines like "data: {...}\n\n", ending in "data: [DONE]"
    if "data:" in raw_text:
        pieces = []
        for line in raw_text.splitlines():
            line = line.strip()
            if not line.startswith("data:"):
                continue
            payload = line[len("data:"):].strip()
            if payload == "[DONE]" or not payload:
                continue
            try:
                chunk = json.loads(payload)
                delta = chunk["choices"][0].get("delta", {})
                content = delta.get("content")
                if content:
                    pieces.append(content)
            except Exception:
                continue
        if pieces:
            return "".join(pieces)

    return None


def call_kimi(prompt, max_retries=MAX_RETRIES):
    headers = {
        "Authorization": f"Bearer {UVARC_API_KEY}",
        "Content-Type": "application/json",
    }
    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "stream": False,
    }

    for attempt in range(max_retries):
        try:
            with httpx.Client(timeout=120.0) as client:
                resp = client.post(UVARC_CHAT_ENDPOINT, headers=headers, json=body)

            _call_counter["n"] += 1
            if _call_counter["n"] <= DEBUG_FIRST_N:
                print(f"\n--- RAW RESPONSE (call {_call_counter['n']}, status {resp.status_code}) ---")
                print(resp.text[:2000])
                print("--- END RAW RESPONSE ---\n")

            if resp.status_code == 401:
                sys.exit("ERROR: 401 Unauthorized — check that UVARC_GenAI_API is a valid, non-expired key.")

            resp.raise_for_status()
            text = _extract_text_from_response(resp.text)
            if text is not None:
                return text.strip()

            print("  Could not parse response into text — see raw output above. Retrying...")
            wait = 10
        except httpx.HTTPStatusError as e:
            err = str(e).lower()
            wait = 10 * (2 ** attempt) if ("rate" in err or "limit" in err or "503" in err) else 10
            print(f"  HTTP error: {e} — waiting {wait}s (attempt {attempt+1}/{max_retries})")
        except Exception as e:
            wait = 10
            print(f"  Error: {e} — waiting {wait}s (attempt {attempt+1}/{max_retries})")

        time.sleep(wait)

    print("  Max retries exceeded — defaulting to empty response")
    return ""


def parse_output(output):
    cleaned = re.sub(r"```(?:json)?", "", output).strip()
    try:
        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if match:
            data = json.loads(match.group())
            label = int(data.get("label", 0))
            label = 1 if label != 0 else 0
            subcategory = data.get("subcategory", label)
            reason = str(data.get("label_reason", "")).strip()
            return label, subcategory, reason
    except (json.JSONDecodeError, ValueError, TypeError):
        pass

    fallback_match = re.search(r"[01]", cleaned)
    label = int(fallback_match.group()) if fallback_match else 0
    return label, label, "PARSE_FAILURE — raw output could not be parsed as JSON"

In [ ]:
# ── Load test set + demo pool, build stratified few-shot examples, run ──────
test_df = pd.read_csv(TEST_INPUT_PATH)
print(f"Test set (classified this run): {len(test_df)} posts from {TEST_INPUT_PATH}")


fewshot_source_df = pd.read_csv(FEWSHOT_SOURCE_PATH)
print(f"Few-shot demo pool: {len(fewshot_source_df)} posts from {FEWSHOT_SOURCE_PATH}")

# Sanity check: demo pool and test set should never overlap on id
overlap = set(fewshot_source_df[ID_COL]) & set(test_df[ID_COL])
assert not overlap, f"Leakage: {len(overlap)} ids appear in both demo pool and test set: {overlap}"

# NOTE: groupby().apply() drops the grouping column on newer pandas
# (observed on pandas 3.0.2) — sample via index instead to avoid that.
demo_idx = []
for _label, _group in fewshot_source_df.groupby(LABEL_COL):
    demo_idx.extend(
        _group.sample(n=min(FEWSHOT_PER_CLASS, len(_group)), random_state=FEWSHOT_RANDOM_STATE).index.tolist()
    )
fewshot_df = fewshot_source_df.loc[demo_idx].reset_index(drop=True)
print(f"Selected {len(fewshot_df)} few-shot demo examples "
      f"({FEWSHOT_PER_CLASS} per class, random_state={FEWSHOT_RANDOM_STATE}):")
print(fewshot_df[LABEL_COL].value_counts())

fewshot_text = create_fewshot_examples_text(fewshot_df)

rows = list(test_df.iterrows())  # [(orig_index, row), ...]


def process_row(item):
    orig_index, row = item
    text = str(row[TEXT_COL])
    prompt = build_few_shot_prompt(text, fewshot_text)

    raw_output = call_kimi(prompt)
    predicted_label, predicted_subcategory, label_reason = parse_output(raw_output)

    return orig_index, {
        ID_COL: row.get(ID_COL, orig_index),
        TEXT_COL: text,
        "predicted_label": predicted_label,
        "predicted_subcategory": predicted_subcategory,
        "label_reason": label_reason,
        "raw_output": raw_output,
    }


results_by_index = {}
completed = 0
with ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
    futures = [pool.submit(process_row, item) for item in rows]
    for fut in as_completed(futures):
        orig_index, result = fut.result()
        results_by_index[orig_index] = result
        completed += 1
        print(f"[{completed}/{len(rows)}] id={result[ID_COL]} pred={result['predicted_label']} "
              f"(subcat {result['predicted_subcategory']})")

# Preserve original row order in the output file
results_df = pd.DataFrame([results_by_index[idx] for idx, _ in rows])
results_df.to_csv(OUTPUT_PATH, index=False)

n_parse_failures = (results_df["label_reason"] == "PARSE_FAILURE — raw output could not be parsed as JSON").sum()
print(f"\nSaved: {OUTPUT_PATH}")
print(f"Predicted label distribution:\n{results_df['predicted_label'].value_counts()}")
if n_parse_failures:
    print(f"WARNING: {n_parse_failures} rows had parse failures — check raw_output for those ids.")


In [ ]:
# ── Diagnose + retry failed rows (run this AFTER the main classification cell) ──
results_df = pd.read_csv(OUTPUT_PATH)  # preds_zeroshot.csv

failed_mask = results_df["label_reason"] == "PARSE_FAILURE — raw output could not be parsed as JSON"
failed_df = results_df[failed_mask]

# Split by whether raw_output is empty (retries exhausted) vs non-empty (model
# responded but output wasn't valid JSON — a different problem, not a network issue)
raw_empty = failed_df["raw_output"].isna() | (failed_df["raw_output"].astype(str).str.strip() == "")
network_failures = failed_df[raw_empty]
genuine_parse_failures = failed_df[~raw_empty]

print(f"Total failed: {len(failed_df)} / {len(results_df)}")
print(f"  Network (retries exhausted, empty raw_output): {len(network_failures)}")
print(f"  Genuine parse failures (model responded, bad format): {len(genuine_parse_failures)}")

if len(genuine_parse_failures) > 0:
    print("\nGenuine parse failures — inspect these raw_output values, may need a prompt tweak:")
    for _, row in genuine_parse_failures.iterrows():
        print(f"  id={row[ID_COL]}: {str(row['raw_output'])[:200]}")

# ── Retry only the network-exhausted rows, at lower concurrency ─────────────
RETRY_CONCURRENCY = 2  # lower than the original run, to reduce dropped connections

if len(network_failures) > 0:
    print(f"\nRetrying {len(network_failures)} network-failed rows at concurrency={RETRY_CONCURRENCY}...")

    retry_rows = list(network_failures.iterrows())

    def process_retry(item):
        idx, row = item
        text = str(row[TEXT_COL])
        prompt = build_few_shot_prompt(text, fewshot_text)
        raw_output = call_kimi(prompt)
        predicted_label, predicted_subcategory, label_reason = parse_output(raw_output)
        return row[ID_COL], {
            "predicted_label": predicted_label,
            "predicted_subcategory": predicted_subcategory,
            "label_reason": label_reason,
            "raw_output": raw_output,
        }

    retry_results = {}
    completed = 0
    with ThreadPoolExecutor(max_workers=RETRY_CONCURRENCY) as pool:
        futures = [pool.submit(process_retry, item) for item in retry_rows]
        for fut in as_completed(futures):
            row_id, result = fut.result()
            retry_results[row_id] = result
            completed += 1
            print(f"  [{completed}/{len(retry_rows)}] id={row_id} pred={result['predicted_label']} "
                  f"reason={result['label_reason'][:40]}")

    # Patch retried results back into results_df
    for row_id, result in retry_results.items():
        mask = results_df[ID_COL] == row_id
        for col, val in result.items():
            results_df.loc[mask, col] = val

    still_failed = (results_df["label_reason"] == "PARSE_FAILURE — raw output could not be parsed as JSON").sum()
    print(f"\nAfter retry: {still_failed} / {len(results_df)} still failed "
          f"(was {len(failed_df)})")

    results_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved updated {OUTPUT_PATH}")
else:
    print("\nNo network-exhausted rows to retry.")
